BG + Product + water mark + opacity control + PNG

In [ ]:
#@title 🖼️ Image Merging & Watermarking Tool (Opacity & Resize)
#@markdown ### Step 1: Run this cell to start.
#@markdown 👉 Apne folder paths aur settings neeche set karein.

import os
from google.colab import drive
from PIL import Image, ImageEnhance
from tqdm.notebook import tqdm

# --- 1. MOUNT GOOGLE DRIVE ---
if not os.path.exists('/content/drive/My Drive'):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted.")

# --- 2. SETTINGS (Yahan Values Change Karein) ---

# Folders ke Paths (Drive ke hisaab se sahi path likhein)
bg_folder_path = '' #@param {type:"string"}
product_folder_path = '' #@param {type:"string"}
watermark_folder_path = '' #@param {type:"string"}
output_folder_path = '' #@param {type:"string"}

# Image Size Settings
target_width = 1080 #@param {type:"integer"}
target_height = 1080 #@param {type:"integer"}

# Watermark Settings
watermark_opacity = 30 #@param {type:"slider", min:0, max:100, step:5}

# --- FUNCTIONS ---

def change_opacity(img, opacity):
    """
    Changes the opacity of an image.
    opacity: integer from 0 to 100
    """
    # Image ko RGBA convert karein taake alpha channel mile
    img = img.convert("RGBA")

    # Agar opacity 100 hai to kuch change na karein
    if opacity == 100:
        return img

    # Calculate factor (e.g. 30% -> 0.3)
    factor = opacity / 100.0

    # Alpha channel alag karein
    r, g, b, a = img.split()

    # Alpha channel ki values ko factor se multiply karein
    a = a.point(lambda i: i * factor)

    # Wapis merge karein
    img.putalpha(a)
    return img

def get_first_image_from_folder(folder_path):
    """Folder mein se pehli image file dhoondta hai."""
    if not os.path.exists(folder_path):
        return None
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    if files:
        return os.path.join(folder_path, files[0])
    return None

def process_images():
    # Output folder banayein agar nahi hai
    if not os.path.exists(output_folder_path):
        os.makedirs(output_folder_path)
        print(f"📁 Created output folder: {output_folder_path}")

    # --- Step A: Load Background ---
    bg_file = get_first_image_from_folder(bg_folder_path)
    if not bg_file:
        print("❌ Error: 1st Folder (Background) mein koi image nahi mili!")
        return

    print(f"🎨 Loading Background: {os.path.basename(bg_file)}")
    background_orig = Image.open(bg_file).convert("RGBA")
    # Resize background to target size
    background_resized = background_orig.resize((target_width, target_height), Image.LANCZOS)

    # --- Step B: Load Watermark & Set Opacity ---
    wm_file = get_first_image_from_folder(watermark_folder_path)
    watermark_ready = None

    if wm_file:
        print(f"©️ Loading Watermark: {os.path.basename(wm_file)} with {watermark_opacity}% Opacity")
        wm_orig = Image.open(wm_file).convert("RGBA")

        # Resize Watermark to target size (Fit to canvas)
        wm_resized = wm_orig.resize((target_width, target_height), Image.LANCZOS)

        # Apply Opacity
        watermark_ready = change_opacity(wm_resized, watermark_opacity)
    else:
        print("⚠️ Warning: 3rd Folder mein Watermark nahi mila. Process bina watermark ke chalega.")

    # --- Step C: Process Products ---
    product_files = [f for f in os.listdir(product_folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]

    if not product_files:
        print("❌ Error: 2nd Folder (Products) khali hai!")
        return

    print(f"\n🚀 Starting processing for {len(product_files)} products...")

    for filename in tqdm(product_files, desc="Processing Images", unit="img"):
        try:
            # 1. Background copy karein (Base Layer)
            final_canvas = background_resized.copy()

            # 2. Product Load aur Resize karein (Middle Layer)
            product_path = os.path.join(product_folder_path, filename)
            product_img = Image.open(product_path).convert("RGBA")
            product_resized = product_img.resize((target_width, target_height), Image.LANCZOS)

            # Product ko Background ke ooper paste karein
            # (0,0) position hai, aur third argument mask hai taake transparency maintain rahe
            final_canvas.paste(product_resized, (0, 0), product_resized)

            # 3. Watermark Paste karein (Top Layer)
            if watermark_ready:
                final_canvas.paste(watermark_ready, (0, 0), watermark_ready)

            # 4. Save Image (Format PNG, Name same as product)
            # Extension change karke .png kar rahe hain
            file_name_no_ext = os.path.splitext(filename)[0]
            save_path = os.path.join(output_folder_path, f"{file_name_no_ext}.png")

            final_canvas.save(save_path, "PNG")

        except Exception as e:
            print(f"❌ Failed processing {filename}: {e}")

    print("\n✅ All Done! Images saved in Output folder.")

# --- RUN THE PROCESS ---
process_images()